<div dir="rtl" style="text-align: right; line-height: 1.9; font-family: 'Segoe UI', Tahoma, Arial, sans-serif; font-size: 16px;">

**[🡨 بازگشت به فصل هشتم (قیمت‌گذاری آپشن‌ها)](Fasl_8_Masterclass.ipynb)**

# 🎓 مسترکلاس مهندسی مالی تصادفی با پایتون
## فصل ۹: معادلات دیفرانسیل با مشتقات جزئی و روش تفاضل محدود (PDE & FDM)

---
### 🎯 هدف این نوت‌بوک
در فصل قبل، قیمت آپشن‌ها را با تولید هزاران مسیر تصادفی (مونت‌کارلو) به دست آوردیم. این روش اگرچه قدرتمند است، اما برای محاسبات دقیق یونانی‌ها (Greeks) یا آپشن‌های آمریکایی، کُند و پر از نویز است.

**قضیه فاینمن-کاک (Feynman-Kac Theorem):** این قضیه شاهراهی بین دنیای احتمالات (فرآیندهای تصادفی) و دنیای آنالیز ریاضی (معادلات دیفرانسیل قطعی) است. بر اساس این قضیه، امید ریاضی یک فرآیند تصادفی (قیمت آپشن)، جواب یک **معادله دیفرانسیل با مشتقات جزئی (PDE)** است.

معادله PDE بلک-شولز به شکل زیر است:
$$ \frac{\partial V}{\partial t} + \frac{1}{2}\sigma^2 S^2 \frac{\partial^2 V}{\partial S^2} + r S \frac{\partial V}{\partial S} - rV = 0 $$

در این مسترکلاس می‌آموزیم:
1. **گسسته‌سازی فضا و زمان (Discretization):** تبدیل فضای پیوسته به یک شبکه (Grid) محاسباتی.
2. **روش ضمنی (Implicit FDM):** تبدیل معادله دیفرانسیل به یک دستگاه معادلات خطی با ماتریس‌های سه‌قطری (Tridiagonal Matrices).
3. **شرایط مرزی و نهایی (Boundary & Terminal Conditions):** اعمال منطق مالی (Payoff) به مرزهای شبکه محاسباتی.
4. **حل و مصورسازی سه‌بعدی:** حل ماتریس‌ها با تجزیه LU و رسم رویه قیمت اختیار فروش (Put Option).

</div>

In [ ]:
# Install necessary packages for Chapter 9
!pip install scipy numpy pandas matplotlib seaborn

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import diags
from scipy.linalg import lu_factor, lu_solve
from abc import ABC, abstractmethod

plt.style.use("seaborn-v0_8-darkgrid")
print("\n--- Setup Complete! Libraries for Finite Difference Methods are loaded. ---")

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 16px;">

### 📚 بخش ۱: ریاضیات روش تفاضل محدود (FDM Mathematics)

برای حل PDE بلک-شولز با کامپیوتر، نمی‌توانیم از مشتقات پیوسته استفاده کنیم. ما زمان ($t$) و قیمت ($S$) را به گام‌های کوچک گسسته‌سازی می‌کنیم:
* گام‌های قیمتی: $\Delta S = \frac{S_{max} - S_{min}}{N}$
* گام‌های زمانی: $\Delta t = \frac{T}{M}$

در شبکه محاسباتی ما، مقدار آپشن در قیمت $i\Delta S$ و زمان $j\Delta t$ با $V_i^j$ نشان داده می‌شود.
تقریب مشتقات با استفاده از بسط تیلور (Central & Backward Differences):
$$ \frac{\partial V}{\partial t} \approx \frac{V_i^{j} - V_i^{j-1}}{\Delta t} $$
$$ \frac{\partial V}{\partial S} \approx \frac{V_{i+1}^{j-1} - V_{i-1}^{j-1}}{2\Delta S} $$
$$ \frac{\partial^2 V}{\partial S^2} \approx \frac{V_{i+1}^{j-1} - 2V_i^{j-1} + V_{i-1}^{j-1}}{\Delta S^2} $$

اگر این تقریب‌ها را در معادله اصلی جایگذاری کنیم، به یک معادله بازگشتی می‌رسیم که می‌توان آن را به فرم ماتریسی $A \cdot V^{j-1} = V^j$ نوشت. ماتریس $A$ یک **ماتریس سه‌قطری (Tridiagonal)** است.

</div>

In [ ]:
# --- 1. Base Core for PDE Solvers & 3D Visualization ---

class SecondOrderFDMSolverTemplate(ABC):
    """
    Base architecture for solving 2nd-order PDEs on a discrete 2D Grid (Space x Time).
    """
    def __init__(self, x_min, x_max, T, M, N, func_name="V", space_var_name="S", time_var_name="t"):
        self._M = M  # Number of time steps
        self._N = N  # Number of space steps
        self._x_min = x_min
        self._x_max = x_max
        self._T = T

        self._δx = (x_max - x_min) / N
        self._δt = T / M

        # Grid creation
        self._x = np.linspace(x_min, x_max, N+1)
        self._t = np.linspace(0, T, M+1)

        # U matrix holds the solution: rows=space, cols=time
        self._u = np.zeros((N+1, M+1), dtype=np.float64)

        self._func_name = func_name
        self._space_var_name = space_var_name
        self._time_var_name = time_var_name

    @abstractmethod
    def _apply_initial_or_terminal_conditions(self): ...

    @abstractmethod
    def _apply_boundary_conditions(self, j): ...

    @abstractmethod
    def solve(self): ...

    def plot_solution(self):
        """Renders the solved PDE grid as a stunning 3D surface."""
        plt.style.use("seaborn-v0_8-darkgrid")
        fig = plt.figure(figsize=(12, 7))
        ax = fig.add_subplot(111, projection="3d")

        # Meshgrid for 3D plot
        T_grid, X_grid = np.meshgrid(self._t, self._x)

        surf = ax.plot_surface(T_grid, X_grid, self._u, cmap=plt.cm.viridis,
                               rstride=1, cstride=1, edgecolor="none", alpha=0.9)

        ax.set_xlabel(f'Time ({self._time_var_name})')
        ax.set_ylabel(f'Asset Price ({self._space_var_name})')
        ax.set_zlabel(f'Option Value ({self._func_name})')
        ax.set_title(f"PDE Solution via Finite Difference Method: {self._func_name}({self._space_var_name}, {self._time_var_name})")
        fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10)
        plt.show()

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 16px;">

### 📚 بخش ۲: الگوریتم روش ضمنی (Implicit FDM Method)

در FDM ما دو رویکرد اصلی داریم:
1. **روش صریح (Explicit):** محاسبه لایه زمانی قبل صرفاً با جایگذاری مقادیر لایه فعلی. این روش ساده است اما از نظر ریاضی **ناپایدار (Unstable)** است و در صورت انتخاب گام‌های اشتباه، خروجی منفجر (کد بی‌نهایت) می‌شود.
2. **روش ضمنی (Implicit):** لایه زمانی قبل به صورت حل یک دستگاه معادلات (ماتریس) به دست می‌آید. این روش **کاملاً پایدار (Unconditionally Stable)** است.

در رویکرد ضمنی، ما به معادله ماتریسی $A \cdot u^{j-1} = u^j + B$ می‌رسیم. ماتریس $A$ سه‌قطری است (قطر اصلی `d`، قطر پایین `a`، قطر بالا `c`).
برای حل سریع این ماتریس غول‌پیکر (مثلاً ۱۰۰۰ در ۱۰۰۰) در هر گام زمانی، به جای محاسبه معکوس ماتریس ($A^{-1}$)، از الگوریتم **تجزیه LU (LU Factorization)** با استفاده از توابع `scipy.linalg` بهره می‌بریم.

</div>

In [ ]:
# --- 2. Implicit Solver Core Engine ---

class HeatEquationImplicitFDMSolverTemplate(SecondOrderFDMSolverTemplate):
    """
    Implicit Finite Difference Solver.
    Solves A * u^{j-1} = u^{j} backward in time (from T to 0).
    """
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    @abstractmethod
    def _a(self, i): ...  # Sub-diagonal coefficients

    @abstractmethod
    def _c(self, i): ...  # Super-diagonal coefficients

    @abstractmethod
    def _d(self, i): ...  # Main diagonal coefficients

    def solve(self):
        # 1. Initialize Terminal Conditions (e.g. Option Payoff at Expiry T)
        self._apply_initial_or_terminal_conditions()

        # 2. Build the Tridiagonal Matrix A components
        a = np.zeros(self._N - 1, dtype=np.float64)
        c = np.zeros(self._N - 1, dtype=np.float64)
        d = np.zeros(self._N - 1, dtype=np.float64)

        for i in range(1, self._N):
            a[i-1] = self._a(i)
            d[i-1] = self._d(i)
            c[i-1] = self._c(i)

        # Create sparse Tridiagonal Matrix A
        A = diags([a[1:], d, c[:-1]], [-1, 0, 1], shape=(self._N-1, self._N-1), format="csr").toarray()

        # Pre-compute LU Factorization for extreme speed inside the loop
        lu, piv = lu_factor(A)

        # 3. Time loop: Step backwards from T (index M) down to 0
        for j in range(self._M, 0, -1):
            # Apply Boundary Conditions for current step
            self._apply_boundary_conditions(j-1)

            # Extract interior points from known future state u^j
            B_vec = self._u[1:-1, j].copy()

            # Adjust the ends of B_vec based on boundary conditions
            B_vec[0] -= self._a(1) * self._u[0, j-1]
            B_vec[-1] -= self._c(self._N-1) * self._u[-1, j-1]

            # Solve A * u^{j-1} = B_vec using LU Solver
            self._u[1:-1, j-1] = lu_solve((lu, piv), B_vec)

        return self

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 16px;">

### 📚 بخش ۳: حل معادله بلک-شولز برای اختیار فروش (Put Option)

اکنون مدل عمومی FDM خود را به معادله بلک-شولز متصل می‌کنیم.
در معادله دیفرانسیل: $\frac{\partial V}{\partial t} + \frac{1}{2}\sigma^2 S^2 \frac{\partial^2 V}{\partial S^2} + r S \frac{\partial V}{\partial S} - rV = 0$

با گسسته‌سازی، ضرایب ماتریس سه‌قطری ($a_i, c_i, d_i$) برای سطر $i$ام شبکه قیمتی (جایی که $S_i = i\Delta S$) به شکل زیر استخراج می‌شوند:
* **قطر پایین:** $a_i = \frac{1}{2} \Delta t (r i - \sigma^2 i^2)$
* **قطر اصلی:** $d_i = 1 + \Delta t (\sigma^2 i^2 + r)$
* **قطر بالا:** $c_i = -\frac{1}{2} \Delta t (r i + \sigma^2 i^2)$

**شرایط مرزی (Boundary & Terminal Conditions):**
1. **شرط نهایی (در زمان $T$):** ارزش آپشن در سررسید برابر است با تابع Payoff. برای اختیار فروش (Put): $V_i^M = \max(K - S_i, 0)$
2. **شرط مرز پایین ($S=0$):** اگر قیمت سهام صفر شود، ارزش Put برابر حداکثر است که به ارزش فعلی تنزیل می‌شود: $V_0^j = K e^{-r(T - j\Delta t)}$
3. **شرط مرز بالا ($S \to \infty$):** اگر قیمت سهام خیلی بالا رود، حق فروش در قیمت $K$ کاملاً بی‌ارزش است: $V_N^j = 0$

</div>

In [ ]:
# --- 3. Black-Scholes PDE Solver for Put Option ---

class BlackScholesPutOptionsFDMSolver(HeatEquationImplicitFDMSolverTemplate):
    """
    Solves the Black-Scholes PDE for a European Put Option using Implicit FDM.
    """
    def __init__(self, r, σ, K, T_expiry, S_max, M, N):
        self._r = r
        self._σ = σ
        self._K = K

        super().__init__(x_min=0.0, x_max=S_max, T=T_expiry, M=M, N=N,
                         func_name="Put Option Value (V)", space_var_name='Stock Price (S)', time_var_name='t')

    # --- 1. Matrix Coefficients derived from BS PDE ---
    def _a(self, i):
        # Sub-diagonal: 0.5 * Δt * (ri - σ^2 i^2)
        return 0.5 * self._δt * (self._r * i - (self._σ**2) * (i**2))

    def _d(self, i):
        # Main-diagonal: 1 + Δt * (σ^2 i^2 + r)
        return 1.0 + self._δt * ((self._σ**2) * (i**2) + self._r)

    def _c(self, i):
        # Super-diagonal: -0.5 * Δt * (ri + σ^2 i^2)
        return -0.5 * self._δt * (self._r * i + (self._σ**2) * (i**2))

    # --- 2. Logic & Financial Conditions ---
    def _apply_initial_or_terminal_conditions(self):
        # Terminal Condition at expiry t=T (index M)
        # Put Payoff = max(K - S, 0)
        for i in range(self._N + 1):
            self._u[i, self._M] = np.maximum(self._K - self._x[i], 0.0)

    def _apply_boundary_conditions(self, j):
        # At S=0 (index i=0), the option is almost certainly exercised.
        # Value is the discounted strike price: K * e^{-r * time_to_maturity}
        time_to_maturity = self._T - (j * self._δt)
        self._u[0, j] = self._K * np.exp(-self._r * time_to_maturity)

        # At S=S_max (index i=N), the Put option is deeply out-of-the-money.
        # Value is 0.
        self._u[self._N, j] = 0.0

print("Black-Scholes FDM Solver Architecture is Initialized.")

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 16px;">

### 📚 بخش ۴: اجرای سیستم و مشاهده رویه قیمت (Execution)

اکنون مدل خود را با پارامترهای واقعی (مثلاً $r=5\%$, $\sigma=20\%$, قیمت توافقی $K=100$) اجرا می‌کنیم. ما یک شبکه قیمتی از ۰ تا ۲۰۰ دلار (با ۵۰ گام) و زمان ۱ سال (با ۵۰۰ گام) ایجاد کرده‌ایم. الگوریتم با سرعت بالا ماتریس را از $T$ به سمت زمان حال ($t=0$) حل کرده و نتیجه را ترسیم می‌کند.

</div>

In [ ]:
def run_fdm_black_scholes():
    print("Building the Grid and solving PDE backwards in time via LU Factorization...")

    # Parameters
    strike_K = 100.0
    risk_free_r = 0.05
    volatility_sigma = 0.20
    time_T = 1.0       # 1 Year to expiry

    # Grid settings
    S_max = 200.0      # Maximum stock price to consider in boundary
    time_steps_M = 500 # Smooth temporal resolution
    space_steps_N = 50 # Spatial resolution

    # Instantiate and Solve
    bs_put_solver = BlackScholesPutOptionsFDMSolver(
        r=risk_free_r,
        σ=volatility_sigma,
        K=strike_K,
        T_expiry=time_T,
        S_max=S_max,
        M=time_steps_M,
        N=space_steps_N
    )

    bs_put_solver.solve()
    print("PDE Solved successfully!")

    # Render stunning 3D Surface
    bs_put_solver.plot_solution()

run_fdm_black_scholes()

<div dir="rtl" style="text-align: right; line-height: 1.9; font-size: 16px;">

---
### 🏁 نتیجه‌گیری فصل ۹
در این فصل، قدرت واقعی ریاضیات محاسباتی را مشاهده کردید:
1. ما توانستیم یک معادله دیفرانسیل جزئی (PDE) پیچیده که توصیف‌گر سیستم‌های مالی تصادفی است را به یک شبکه ساده اعداد (Grid) تبدیل کنیم.
2. با استفاده از **روش ضمنی و جبر خطی (LU Factorization)**، این شبکه را از زمان آینده (سررسید) به سمت امروز حل کردیم تا به قیمت قطعی و بدون نویز برسیم.
3. این روش (FDM) در بانک‌های سرمایه‌گذاری برای قیمت‌گذاری **آپشن‌های آمریکایی (American Options)** استفاده می‌شود، زیرا برخلاف روش مونت‌کارلو (که فقط روبه‌جلو می‌رود)، FDM روبه‌عقب حرکت کرده و به راحتی می‌تواند «قابلیت اعمال زودهنگام» (Early Exercise) را در هر گره محاسباتی ارزیابی کند.

در فصل پایانی (فصل ۱۰)، از حوزه قیمت‌گذاری خارج شده و وارد دنیای **مدیریت ریسک و بهینه‌سازی سبد سرمایه‌گذاری (Portfolio Optimization)** بر اساس تئوری مدرن پورتفولیو (Markowitz) خواهیم شد.

</div>